<a href="https://colab.research.google.com/github/syahidmid/google-colab/blob/main/wordpress/case-study/fix_duplicate_title_v2_retry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ Fix Duplicate Title v2 (from Failed Log) - Bali Touristic

Versi retry khusus untuk post yang **gagal** di run sebelumnya.

**Source:** Upload `fix_log_failed.csv` — hanya post di dalam file ini yang akan diproses.

**Fix:** Hapus semua `<h1>` di manapun letaknya di dalam content.

| Kondisi | Aksi |
|---|---|
| Semua `<h1>` di content | **Dihapus total** |
| Tidak ada H1 | Skip |

In [ ]:
import requests
import csv
import time
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup
from IPython.display import display, HTML

try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    userdata = None

print("✅ Library berhasil diimport.")

In [ ]:
# ─────────────────────────────────────────────
# 📂 Upload fix_log_failed.csv
# ─────────────────────────────────────────────
print("📂 Upload file fix_log_failed.csv ...\n")

if IN_COLAB:
    uploaded = files.upload()
    csv_filename = list(uploaded.keys())[0]
else:
    csv_filename = input("Path file CSV: ").strip()

# Baca post IDs dari CSV
failed_posts = []
with open(csv_filename, mode="r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    reader.fieldnames = [h.strip() for h in reader.fieldnames]
    for row in reader:
        failed_posts.append({
            "id": str(row.get("id", "")).strip(),
            "title": str(row.get("title", "")).strip(),
            "url": str(row.get("url", "")).strip(),
        })

# Filter baris kosong
failed_posts = [p for p in failed_posts if p["id"]]

print(f"✅ File '{csv_filename}' berhasil dibaca.")
print(f"📋 Total post dari log gagal: {len(failed_posts)}")
print("\nContoh 5 pertama:")
for p in failed_posts[:5]:
    print(f"  [{p['id']}] {p['title'][:55]}")

In [ ]:
# ─────────────────────────────────────────────
# 🔑 Load Credentials
# ─────────────────────────────────────────────
wordpress_url = "https://www.balitouristic.com/wp-json/wp/v2/posts"

if userdata:
    username = userdata.get('admin_bali')
    password = userdata.get('pass_bali')
else:
    username = None
    password = None

if not username or not password:
    raise ValueError("❌ Kredensial tidak ditemukan di userdata.")

auth = HTTPBasicAuth(username, password)
print("✅ Kredensial berhasil dimuat.")

In [ ]:
# ─────────────────────────────────────────────
# 🔧 Fungsi: Hapus semua H1 di content
# ─────────────────────────────────────────────
def remove_all_h1(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    h1_tags = soup.find_all("h1")
    count = len(h1_tags)
    for tag in h1_tags:
        tag.decompose()
    return str(soup), count

print("✅ Fungsi remove_all_h1 siap.")

In [ ]:
# ─────────────────────────────────────────────
# 🚀 Main Loop: Fetch per ID → Fix → Update
# ─────────────────────────────────────────────
log_success = []
log_failed  = []
log_skipped = []

total = len(failed_posts)

print(f"🚀 Memulai retry fix H1 untuk {total} post dari log gagal...\n")
print("=" * 65)

for idx, post in enumerate(failed_posts, start=1):
    pid        = post["id"]
    post_url   = post["url"]
    post_title = post["title"][:50]

    print(f"[{idx:>4}/{total}] 🔍 ID {pid} | {post_title}")

    # Fetch content langsung per ID
    try:
        resp = requests.get(
            f"{wordpress_url}/{pid}",
            auth=auth,
            params={"_fields": "id,title,link,content"}
        )
    except Exception as e:
        print(f"         ❌ Gagal fetch: {e}")
        log_failed.append({"id": pid, "title": post_title, "url": post_url, "reason": str(e)})
        continue

    if resp.status_code != 200:
        print(f"         ❌ HTTP {resp.status_code} saat fetch")
        log_failed.append({"id": pid, "title": post_title, "url": post_url, "reason": f"Fetch HTTP {resp.status_code}"})
        continue

    data        = resp.json()
    content_obj = data.get("content", {})
    raw_content = content_obj.get("raw", "") or content_obj.get("rendered", "") if isinstance(content_obj, dict) else ""

    if not raw_content:
        print(f"         ⬜ Content kosong, skip.")
        log_skipped.append({"id": pid, "title": post_title, "url": post_url, "reason": "Content kosong"})
        continue

    # Hapus semua H1
    fixed_content, h1_count = remove_all_h1(raw_content)

    if h1_count == 0:
        print(f"         ⬜ Tidak ada H1 ditemukan, skip.")
        log_skipped.append({"id": pid, "title": post_title, "url": post_url, "reason": "Tidak ada H1"})
        continue

    # Update ke WordPress
    try:
        update_resp = requests.post(
            f"{wordpress_url}/{pid}",
            auth=auth,
            json={"content": fixed_content}
        )
    except Exception as e:
        print(f"         ❌ Gagal update: {e}")
        log_failed.append({"id": pid, "title": post_title, "url": post_url, "reason": str(e)})
        continue

    if update_resp.status_code in (200, 201):
        print(f"         ✅ {h1_count} H1 dihapus!")
        log_success.append({"id": pid, "title": post_title, "url": post_url, "h1_removed": h1_count})
    else:
        print(f"         ❌ HTTP {update_resp.status_code} saat update")
        log_failed.append({"id": pid, "title": post_title, "url": post_url, "reason": f"Update HTTP {update_resp.status_code}"})

    time.sleep(0.3)

print("\n" + "=" * 65)

In [ ]:
# ─────────────────────────────────────────────
# 🎉 SUMMARY LOG
# ─────────────────────────────────────────────
total_h1_removed = sum(r["h1_removed"] for r in log_success)

summary_html = f"""
<div style="font-family: 'Segoe UI', sans-serif; max-width: 700px; margin: 16px 0;">
  <div style="background: linear-gradient(135deg, #1a1a2e, #16213e); border-radius: 16px; padding: 28px; color: white; box-shadow: 0 8px 32px rgba(0,0,0,0.3);">

    <div style="font-size: 22px; font-weight: 700; letter-spacing: 1px; margin-bottom: 4px;">🛠️ DUPLICATE TITLE v2 — RETRY COMPLETE</div>
    <div style="font-size: 12px; color: #718096; margin-bottom: 6px;">Source: fix_log_failed.csv · All H1 removed regardless of position</div>
    <div style="font-size: 13px; color: #a0aec0; margin-bottom: 24px;">balitouristic.com · WordPress REST API</div>

    <div style="display: flex; gap: 16px; flex-wrap: wrap; margin-bottom: 24px;">
      <div style="flex:1; min-width:120px; background: rgba(72,199,142,0.15); border: 1px solid #48c78e; border-radius: 12px; padding: 16px; text-align:center;">
        <div style="font-size: 36px; font-weight: 800; color: #48c78e;">{len(log_success)}</div>
        <div style="font-size: 11px; color: #a0aec0; margin-top: 4px;">BERHASIL DIFIX</div>
      </div>
      <div style="flex:1; min-width:120px; background: rgba(255,107,107,0.15); border: 1px solid #ff6b6b; border-radius: 12px; padding: 16px; text-align:center;">
        <div style="font-size: 36px; font-weight: 800; color: #ff6b6b;">{total_h1_removed}</div>
        <div style="font-size: 11px; color: #a0aec0; margin-top: 4px;">TOTAL H1 DIHAPUS</div>
      </div>
      <div style="flex:1; min-width:120px; background: rgba(160,174,192,0.1); border: 1px solid #718096; border-radius: 12px; padding: 16px; text-align:center;">
        <div style="font-size: 36px; font-weight: 800; color: #a0aec0;">{len(log_skipped)}</div>
        <div style="font-size: 11px; color: #a0aec0; margin-top: 4px;">DILEWATI</div>
      </div>
      <div style="flex:1; min-width:120px; background: rgba(246,173,85,0.15); border: 1px solid #f6ad55; border-radius: 12px; padding: 16px; text-align:center;">
        <div style="font-size: 36px; font-weight: 800; color: #f6ad55;">{len(log_failed)}</div>
        <div style="font-size: 11px; color: #a0aec0; margin-top: 4px;">MASIH GAGAL</div>
      </div>
    </div>

    <div style="background: rgba(255,255,255,0.05); border-radius: 10px; padding: 14px; font-size: 13px; color: #cbd5e0; line-height: 1.9;">
      📋 Total dari log gagal : <b style='color:white'>{total}</b><br>
      ✅ Berhasil difix &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;: <b style='color:#48c78e'>{len(log_success)}</b><br>
      🗑️ Total H1 dihapus &nbsp;: <b style='color:#ff6b6b'>{total_h1_removed}</b><br>
      ⬜ Dilewati (skip) &nbsp;&nbsp;&nbsp;: <b style='color:#a0aec0'>{len(log_skipped)}</b><br>
      ❌ Masih gagal &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;: <b style='color:#f6ad55'>{len(log_failed)}</b>
    </div>

  </div>
</div>
"""
display(HTML(summary_html))

In [ ]:
# ─────────────────────────────────────────────
# 💾 Download Log CSV
# ─────────────────────────────────────────────

if log_success:
    fname = "fix_v2_retry_success.csv"
    with open(fname, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "title", "url", "h1_removed"])
        writer.writeheader()
        writer.writerows(log_success)
    print(f"💾 Log sukses: '{fname}' ({len(log_success)} baris)")
    if IN_COLAB:
        files.download(fname)

if log_failed:
    fname = "fix_v2_retry_failed.csv"
    with open(fname, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "title", "url", "reason"])
        writer.writeheader()
        writer.writerows(log_failed)
    print(f"💾 Log masih gagal: '{fname}' ({len(log_failed)} baris)")
    if IN_COLAB:
        files.download(fname)

if not log_success and not log_failed:
    print("⬜ Tidak ada perubahan yang dilakukan.")